# OLS & BLUE Properties: Monte Carlo Method
This notebook examines the BLUE properties of  OLS estimators.

Part 1: Generates a synthetic dataset using Monte Carlo simulations.

Part 2: Analyses how different sample sizes and error term distributions affect the key properties of OLS estimators.

## Part 1 - Generating the Data

Creation of the DataFrame to study the BLUE propertis of OLS estimators.

The true population function as a DGP is assumed to be $Y_i=\beta_0+\beta X_{i} +\beta_2 X_{i}^2$ with the true known parameters $\beta_0=1$, $\beta_1=-3$ and $\beta_2=2$.

Based on this DGP, the Python code below uses Monte Carlo simulation to gerenerate 10,000 OLS estimators $\hat{\beta_0}$, $\hat{\beta_1}$, $\hat{\beta_2}$ for different sample sizes ($N=15$, $N=50$, $N=200$, $N=1000$) and different distributions of the error term (normal distribution and uniform distribution).

Final results are stored in a DataFrame and exportes as csv and parquet files.

In [22]:
import numpy as np
import pandas as pd
import statsmodels.api as sm

# --- Simulation Parameters ---
N_OBS = [15, 50, 200, 1000]    # Sample size per regression
N_SIMS = 10000                 # Number of Monte Carlo iterations
BETA_0 = 1                     # Intercept
BETA_1 = -3                    # True coefficient for X
BETA_2 = 2                     # True coefficient for X^2


def run_monte_carlo(dist, n_obs, n_sims=N_SIMS):
    b0_estimates = []
    b1_estimates = []
    b2_estimates = []

    for _ in range(n_sims):
        # 1. Generate Independent Variable X
        X = np.linspace(-5, 5, n_obs)
        X_squared = X**2

        # 2. Generate response variables Y with normally and uniformly distributed noise

        if dist == 'normal':
            epsilon = np.random.normal(0, 6, n_obs)
        elif dist == 'uniform':
            epsilon = np.random.uniform(-9, 9, n_obs)
        else:
            raise ValueError("Invalid distribution")
        Y = BETA_0 + BETA_1 * X + BETA_2 * X_squared + epsilon

        # 3. Fit OLS Regression
        X_design = np.column_stack([X, X_squared])
        X_design = sm.add_constant(X_design)
        model = sm.OLS(Y, X_design).fit()

        # 4. Store estimated coefficients
        b0_estimates.append(model.params[0])
        b1_estimates.append(model.params[1])
        b2_estimates.append(model.params[2])

    return b0_estimates, b1_estimates, b2_estimates

# --- Store results in single DataFrames ---
my_dict_norm={}
my_dict_unif={}
for index, obs in enumerate(N_OBS):
    b0_hat_norm, b1_hat_norm, b2_hat_norm = run_monte_carlo("normal", obs, N_SIMS)
    b0_hat_unif, b1_hat_unif, b2_hat_unif = run_monte_carlo("uniform", obs, N_SIMS)

    my_dict_norm[index] = pd.DataFrame({
    'beta_0': b0_hat_norm,
    'beta_1': b1_hat_norm,
    'beta_2': b2_hat_norm,
    'sample_size': obs,
    'dist': 'normal'
    })

    my_dict_unif[index] = pd.DataFrame({
    'beta_0': b0_hat_unif,
    'beta_1': b1_hat_unif,
    'beta_2': b2_hat_unif,
    'sample_size': obs,
    'dist': 'uniform'
    })

# --- Combining all single DataFrames & exporting to files---
df_big_norm=pd.concat([x for x in my_dict_norm.values()],ignore_index=True)
df_big_unif=pd.concat([x for x in my_dict_unif.values()],ignore_index=True)
df_big=pd.concat([df_big_norm,df_big_unif],ignore_index=True)
df_big['sample_size'] = df_big['sample_size'].astype(np.int32)
df_big['dist'] = df_big['dist'].astype("category")
df_big.to_csv("MonteCarlo_BLUE.csv", index=False)
df_big.to_parquet("MonteCarlo_BLUE.parquet")
